# TYK2 research results: supervisor review

This notebook checks the delivered rankings and shows the evidence behind them. It does **not** train the research models. Training data, model weights, and the 555,332-row predictions are large separate artifacts; see [the review guide](REVIEW_TYK2.md) for provenance and the required files. The public quickstart notebook is for running a new project on your own CSVs.

Run this notebook from the repository root after placing the corrected delivery folder under `deliverables/`. The artifact audit deliberately reports missing files and stops before making claims about unavailable data.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

repo = Path.cwd()
if not (repo / 'README.md').exists():
    repo = repo.parent
DELIVERY = repo / 'deliverables' / 'lindsey_corrected_all_models_20260923'
REQUIRED = [
    'delivery_manifest.json',
    'full_library_all_model_rankings.csv.gz',
    'selected_400_updated_constrained_ensemble.csv',
    'model_metrics_all_available.csv',
    'heldout_predictions_all_available.csv',
    'training_curves_all_available.csv',
    'round1_fep_model_comparison.csv',
    'full_pool_distribution_statistics.csv',
    'all_model_distributions.png',
]
inventory = pd.DataFrame([{'file': name, 'present': (DELIVERY / name).is_file(),
                           'bytes': (DELIVERY / name).stat().st_size if (DELIVERY / name).is_file() else None}
                          for name in REQUIRED])
display(inventory)
missing = inventory.loc[~inventory.present, 'file'].tolist()
if missing:
    raise FileNotFoundError(f'Missing delivery files in {DELIVERY}: {missing}. See notebooks/REVIEW_TYK2.md.')
published_manifest = pd.read_csv(repo / 'notebooks' / 'tyk2_artifact_manifest.csv')
hash_checks = []
for record in published_manifest.itertuples(index=False):
    path = DELIVERY / record.file
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    hash_checks.append({'file': record.file, 'size_matches': path.stat().st_size == record.bytes,
                        'sha256_matches': digest.hexdigest().upper() == record.sha256.upper()})
hash_checks = pd.DataFrame(hash_checks)
display(hash_checks)
assert hash_checks[['size_matches', 'sha256_matches']].all().all(), 'Shared artifacts differ from the public manifest.'
manifest = json.loads((DELIVERY / 'delivery_manifest.json').read_text(encoding='utf-8'))
display(pd.Series(manifest, name='manifest'))

## Optional source archive check

Extract the separately shared source ZIP into `supervisor_tyk2_source_20260923/` at the repository root to verify the scripts and methods log against the public source manifest.

In [ ]:
source_dir = repo / 'supervisor_tyk2_source_20260923'
if source_dir.exists():
    source_manifest = pd.read_csv(repo / 'notebooks' / 'tyk2_source_manifest.csv')
    source_checks = []
    for record in source_manifest.itertuples(index=False):
        path = source_dir / record.file
        if not path.is_file():
            source_checks.append({'file': record.file, 'passed': False})
            continue
        digest = hashlib.sha256(path.read_bytes()).hexdigest().upper()
        source_checks.append({'file': record.file, 'passed': path.stat().st_size == record.bytes and digest == record.sha256.upper()})
    display(pd.DataFrame(source_checks))
    assert all(item['passed'] for item in source_checks)
else:
    print('Optional source archive not extracted; result-file checks above are complete.')

## 1. Full-pool integrity

Read the compressed ranking in chunks. Check the molecule count, unique structures, continuous selection ranks, finite predictions, and the expected coverage of the six historical models and eight later models.

In [ ]:
ranked_path = DELIVERY / 'full_library_all_model_rankings.csv.gz'
models = manifest['full_pool_models']
seen = set()
coverage = {name: 0 for name in models}
count = 0
rank_ok = True
finite_ok = True
first_400 = []
for chunk in pd.read_csv(ranked_path, chunksize=50_000, usecols=['smiles', 'selection_rank', *models]):
    n = len(chunk)
    rank_ok &= np.array_equal(chunk.selection_rank.to_numpy(), np.arange(count + 1, count + n + 1))
    if count == 0:
        first_400 = chunk.head(400).smiles.tolist()
    seen.update(chunk.smiles)
    for name in models:
        values = chunk[name].to_numpy(dtype=float)
        coverage[name] += int(np.isfinite(values).sum())
        finite_ok &= not np.isinf(values).any()
    count += n
checks = {'pool_rows': count == 555_332 == manifest['pool_rows'],
          'unique_smiles': len(seen) == count,
          'selection_ranks_1_to_N': rank_ok,
          'no_infinite_predictions': finite_ok,
          'historical_coverage_547490_each': all(coverage[m] == 547_490 for m in models[:6]),
          'later_coverage_555332_each': all(coverage[m] == 555_332 for m in models[6:])}
display(pd.Series(checks, name='passed'))
display(pd.Series(coverage, name='predicted_molecules').to_frame())
assert all(checks.values()), 'Investigate failed full-pool checks before using the ranking.'

## 2. Selection and experimental overlap

The delivered 400 are the first 400 by the constrained consensus's uncertainty-aware selection rank. The round-1 flag identifies molecules already in the 1,109-ligand training set; they should be separated from candidates needing new experimental edges.

In [ ]:
selected = pd.read_csv(DELIVERY / 'selected_400_updated_constrained_ensemble.csv')
assert len(selected) == selected.smiles.nunique() == 400
assert selected.selection_rank.tolist() == list(range(1, 401))
assert selected.smiles.tolist() == first_400
already_measured = selected.in_round1_1109_training.astype(str).str.lower().eq('true')
assert int(already_measured.sum()) == manifest['selected_already_in_round1_1109']
display(pd.Series({'selected': len(selected), 'already_in_round1': int(already_measured.sum()),
                   'not_in_round1': int((~already_measured).sum())}, name='molecules'))
display(selected[['smiles', 'selection_rank', 'Constrained_consensus_DG', 'uncertainty_score',
                  'in_round1_1109_training']].head(15))

## 3. Model performance and training traces

Review the original held-out splits, later fixed-75 tests, and round-1 FEP retrospective separately. The round-1 comparisons involve selected molecules and are not prospective validation. Tree models do not have epoch loss curves; available curves are in the delivery.

In [ ]:
metrics = pd.read_csv(DELIVERY / 'model_metrics_all_available.csv')
heldout = pd.read_csv(DELIVERY / 'heldout_predictions_all_available.csv')
curves = pd.read_csv(DELIVERY / 'training_curves_all_available.csv')
round1 = pd.read_csv(DELIVERY / 'round1_fep_model_comparison.csv')
display(metrics[['model', 'split', 'n', 'rmse', 'mae', 'r2', 'spearman']])
display(round1.sort_values('spearman', ascending=False))
display(curves.groupby(['model', 'split'], dropna=False).size().rename('logged_points').reset_index())
print(f'{len(heldout)} saved held-out prediction rows across {heldout.model_key.nunique()} models')

In [ ]:
figure = DELIVERY / 'all_model_distributions.png'
if figure.exists():
    display(Image(filename=str(figure), width=1100))
else:
    print('Distribution figure unavailable; numeric statistics are still in full_pool_distribution_statistics.csv.')
fig, ax = plt.subplots(figsize=(8, 5))
fixed = metrics.loc[metrics.split.eq('fixed_75') & metrics.spearman.notna()].sort_values('spearman')
ax.barh(fixed.model, fixed.spearman, color='#0072B2')
ax.set(xlabel='Spearman correlation on fixed 75', title='Later models: held-out ranking agreement')
fig.tight_layout()
plt.show()

In [ ]:
# Independently recompute held-out metrics from the saved molecule-level predictions.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
key_map = {'adapted_s2_ridge': 'frozen_ridge', 'adapted_s2_rf': 'molertd_rf',
           'adapted_s2_ecfp_rf': 'molertd_ecfp_rf',
           'adapted_s2_partial_finetune': 'molertd_partial_finetune'}
recomputed = []
for (key, split), group in heldout.groupby(['model_key', 'split']):
    truth = group.true_DG.to_numpy(dtype=float)
    prediction = group.pred_DG.to_numpy(dtype=float)
    recomputed.append({'model_key': key_map.get(key, key), 'split': split, 'n_computed': len(group),
                       'rmse_computed': mean_squared_error(truth, prediction) ** 0.5,
                       'mae_computed': mean_absolute_error(truth, prediction),
                       'r2_computed': r2_score(truth, prediction),
                       'spearman_computed': group.true_DG.corr(group.pred_DG, method='spearman')})
recomputed = pd.DataFrame(recomputed)
comparison = metrics.merge(recomputed, on=['model_key', 'split'], how='left', validate='one_to_one')
for measure in ('rmse', 'mae', 'r2', 'spearman'):
    comparison[f'{measure}_difference'] = (comparison[measure] - comparison[f'{measure}_computed']).abs()
display(comparison[['model', 'split', 'n', 'n_computed', 'rmse_difference', 'mae_difference',
                    'r2_difference', 'spearman_difference']])
assert (comparison.loc[comparison.n_computed.notna(), 'n'] == comparison.loc[comparison.n_computed.notna(), 'n_computed']).all()
assert comparison[['rmse_difference', 'mae_difference', 'r2_difference', 'spearman_difference']].max().max() < 1e-5
print('Saved held-out metrics agree with recomputation (tolerance 1e-5). OOB-only RF summaries have no molecule-level held-out predictions in this package.')

## Interpretation limits

- The six historical model outputs cover 547,490 exact SMILES matches in the cleaned current pool; blank values on the other 7,842 rows are expected.
- The adapted S2 ridge is included for audit. Its extrapolation was unstable and it did not contribute to constrained candidate selection.
- The original 400-ligand selection and the updated 400 are different cohorts; use the round-1 flag when planning additional FEP edges.
- The 1,109-label RF fit was trained using round-1 results. Its predictions cannot be evaluated on those same ligands as independent validation.
- See the source map in the review guide for training and selection details.